In [21]:
import associations
import pandas as pd

df_train = pd.read_csv("./data/clean/train_cleaned.csv")

In [3]:
df_train.head()

,passengerid,homeplanet,cryosleep,cabin,destination,age,vip,roomservice,foodcourt,shoppingmall,spa,vrdeck,name,transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [46]:
import numpy as np
from dython.nominal import associations

target = "transported"
df_train[target] = df_train[target].astype(int)

num_features = df_train.select_dtypes(include=[np.number]).columns.drop(target)
correlations = df_train[num_features].corrwith(df_train[target])
selected_num_features = correlations[abs(correlations) > 0.2].index.tolist()

cat_features = (df_train
                .drop('passengerid', axis=1)
                .select_dtypes(include=['object', 'str', 'bool', 'category']).columns)
assoc = associations(df_train[list(cat_features) + [target]], nominal_columns='all', plot=False)
corr_matrix = assoc['corr']

selected_cat_features = corr_matrix[corr_matrix.loc['transported'] > 0.2].index.tolist()
selected_cat_features = selected_cat_features[:-1]

print("Selected Numeric Features:", selected_num_features)
print("Selected Categorical Features:", selected_cat_features)

selected_features = selected_num_features + selected_cat_features

print(selected_features)

Selected Numeric Features: ['roomservice', 'spa', 'vrdeck']
Selected Categorical Features: ['cryosleep']
['roomservice', 'spa', 'vrdeck', 'cryosleep']


In [45]:
from sklearn.model_selection import train_test_split

X = df_train[selected_features]
y = df_train[target]

cat_features = X.select_dtypes(include=['object']).columns.tolist()
num_features = X.select_dtypes(include=['number']).columns.tolist()

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2,
                                                            random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val,
                                                  test_size=0.25,
                                                  random_state=42,
                                                  stratify=y_train_val)

# split data to using: train:validate:test
# this is use for prediction
print(X_train.shape, X_val.shape, X_test.shape)

# this is use for answer
print(y_train.shape, y_val.shape, y_test.shape)

(5215, 4) (1739, 4) (1739, 4)
(5215,) (1739,) (1739,)


## Train CatBoost
Use feature select by correlation > 0.2 and data split 60:20:20 from above 

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

X_train_cb, X_val_cb, X_test_cb = X_train.copy(), X_val.copy(), X_test.copy()
for d in (X_train_cb, X_val_cb, X_test_cb):
    d[selected_cat_features] = d[selected_cat_features].astype(str)

model = CatBoostClassifier(
    iterations=2000, learning_rate=0.03, depth=5, l2_leaf_reg=5,
    eval_metric="Accuracy", early_stopping_rounds=100,
    random_seed=42, verbose=200, allow_writing_files=False,
)
model.fit(X_train_cb, y_train,
          cat_features=selected_cat_features,
          eval_set=(X_val_cb, y_val))

0:	learn: 0.7587728	test: 0.7492812	best: 0.7492812 (0)	total: 2.27ms	remaining: 4.53s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7630822312
bestIteration = 8

Shrink model to first 9 iterations.


CatBoostClassifier(allow_writing_files=False, depth=5, early_stopping_rounds=100, eval_metric='Accuracy', iterations=2000, l2_leaf_reg=5, learning_rate=0.03, random_seed=42, verbose=200)

In [12]:
def report(name, X, y):
    pred = model.predict(X).astype(int).ravel()
    proba = model.predict_proba(X)[:, 1]
    print(f"{name:10s} acc={accuracy_score(y, pred):.4f}  "
          f"f1={f1_score(y, pred):.4f}  auc={roc_auc_score(y, proba):.4f}")

report("train", X_train_cb, y_train)
report("validation", X_val_cb, y_val)

train      acc=0.7707  f1=0.7807  auc=0.8209
validation acc=0.7631  f1=0.7734  auc=0.8054


## Final evaluation on test (run once)

In [13]:
report("test", X_test_cb, y_test)

test       acc=0.7734  f1=0.7816  auc=0.8135
